# HCI2000 Data Preprocessing
Browser tab orientation (H/V) x tab load (low/high), 16 participants, 24 trials each.

**Pipeline stages**

| Stage | What happens | Output |
|---|---|---|
| 0 | Settings and helper | - |
| 1 | Load the four raw sources | 4 raw DataFrames |
| 2 | Clean each source on its own | 4 clean DataFrames |
| 3 | Cross-source checks (who is present where) | participant coverage table |
| 4 | Combine at the right grain | `trial_level` (384 rows) and `condition_level` (64 rows) |
| 5 | Final validation and export | CSVs in `cleaned/` plus `cleaning_log.csv` |

**Design rules used throughout**
1. **Clean each source separately, merge last.** Each form has its own quirks and its own grain, so a problem is fixed where it lives.
2. **Never merge by row order.** Joins use `participant` (+ `condition`) with `validate=` so a bad key stops the notebook instead of silently duplicating rows.
3. **Flag, don't silently drop.** Every row removed or value changed is written to a cleaning log with a count, so the pipeline can be explained and audited.
4. **Assertions guard the assumptions.** If the study design says 24 trials per participant, the notebook checks it.
5. **No analysis decisions here** (outliers, transformations, tests). Those come after cleaning.

## 0. Setup

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ---- settings: the only cell you should need to edit when switching dummy -> real data ----
DATA_DIR = Path("dummy_data")       # real data: point this at the folder holding the real exports
CODE_PREFIX = "DUMMY"               # real data: "P"
MODE = "test"                       # real data: "main". Pilot / test sessions are filtered out by this
N_PARTICIPANTS = 16
TRIALS_PER_BLOCK = 6
CONDITIONS = ["H-L", "H-H", "V-L", "V-H"]
N_BLOCKS = len(CONDITIONS)
OUT_DIR = Path("cleaned")

ROSTER = [f"{CODE_PREFIX}{n:03d}" for n in range(1, N_PARTICIPANTS + 1)]
TLX_DIMS = ["mental_demand", "physical_demand", "temporal_demand", "performance", "effort", "frustration"]

# ---- cleaning log: every fix/removal is recorded here and shown at the end ----
log = []
def note(stage, issue, n, action):
    """Record one cleaning decision. n = how many rows/values were affected."""
    log.append({"stage": stage, "issue": issue, "n_affected": n, "action": action})
    print(f"[{stage}] {issue}: {n} -> {action}")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Load the raw sources
Four sources, **three different grains** - this is why they are not simply stacked into one file:

| Source | One row = | Expected rows |
|---|---|---|
| `main_data` (one CSV per participant) | one trial | 16 x 24 = 384 |
| block NASA-TLX form | one participant x one condition | 16 x 4 = 64 |
| weighted NASA-TLX form | one participant | 16 |
| screening form | one person | >= 16 (also contains people who were never enrolled) |

### 1.1 main_data.csv (combine the per-participant files)

In [2]:
main_files = sorted((DATA_DIR / "main_data").glob("*_main_data.csv"))
print(f"{len(main_files)} files found")

# Every file must have the same columns. pd.concat would quietly union mismatched columns, so check first.
headers = {f.name: tuple(pd.read_csv(f, nrows=0).columns) for f in main_files}
assert len(set(headers.values())) == 1, "main_data files do not share the same columns"

main_raw = pd.concat(
    (pd.read_csv(f).assign(source_file=f.name) for f in main_files), ignore_index=True
)
print("Raw main_data shape:", main_raw.shape)
main_raw.head(3)

16 files found
Raw main_data shape: (385, 26)


,session_id,mode,participant,group,block,condition,orientation,tab_load,content_set,trial,target_id,target_title,is_repeat,target_task_position,task_tab_count,visible_tab_count,controller_status,measurement_source,retrieval_time_s,total_tab_switches,incorrect_selections,first_selection_correct,no_selection,technical_loss,notes,source_file
0,DUMMY-20261101T090000Z-663470961711004,test,DUMMY001,2,1,H-H,H,high,C,T1,P3,Coral Reefs Support Marine Life,0,7,15,16,completed_auto,controller_accessibility_and_poll,10.304774,3.0,2.0,False,False,False,NaN,DUMMY001_main_data.csv
1,DUMMY-20261101T090000Z-663470961711004,test,DUMMY001,2,1,H-H,H,high,C,T2,P4,Mangroves and Coastal Habitats,0,10,15,16,completed_auto,controller_accessibility_and_poll,10.523085,1.0,0.0,True,False,False,NaN,DUMMY001_main_data.csv
2,DUMMY-20261101T090000Z-663470961711004,test,DUMMY001,2,1,H-H,H,high,C,T3,P5,Mountains and Erosion Patterns,0,13,15,16,completed_auto,controller_accessibility_and_poll,15.109607,3.0,2.0,False,False,False,NaN,DUMMY001_main_data.csv


### 1.2 Block NASA-TLX, weighted NASA-TLX, screening

In [3]:
block_raw    = pd.read_csv(DATA_DIR / "forms" / "block_survey.csv")
weighted_raw = pd.read_csv(DATA_DIR / "forms" / "weighted_tlx.csv")
screen_raw   = pd.read_csv(DATA_DIR / "forms" / "screening.csv")

for name, d in [("block", block_raw), ("weighted", weighted_raw), ("screening", screen_raw)]:
    print(f"{name:10s} raw shape: {d.shape}")

block      raw shape: (64, 11)
weighted   raw shape: (15, 18)
screening  raw shape: (17, 11)


## 2. Clean each source separately

### 2.1 main_data (trial level)
Order matters: **filter mode -> remove exact duplicates -> validate structure -> check missing values**.
Duplicates are removed *before* the structure check, otherwise a duplicated row looks like a 25th trial.

In [4]:
main = main_raw.copy()

# 1) Only sessions of the right mode (drops pilot / test runs that must never reach the analysis)
before = len(main)
main = main[main["mode"] == MODE].copy()
note("main", f"rows with mode != '{MODE}'", before - len(main), "dropped")

# 2) Exact duplicate rows (same file, identical in every column)
n_dup = int(main.duplicated().sum())
main = main.drop_duplicates().reset_index(drop=True)
note("main", "exact duplicate rows", n_dup, "dropped (kept first)")

# 3) Tidy types. The controller writes true/false text; blanks must stay missing, not become False.
for c in ["first_selection_correct", "no_selection", "technical_loss"]:
    main[c] = main[c].map({True: True, False: False, "true": True, "false": False}).astype("boolean")
main["participant"] = main["participant"].str.strip().str.upper()

[main] rows with mode != 'test': 0 -> dropped
[main] exact duplicate rows: 1 -> dropped (kept first)


**Structure checks** - the design says 24 trials per participant, 6 in each of 4 conditions, each (participant, block, trial) exactly once.
`assert` makes the notebook stop if the data break that promise.

In [5]:
key = ["participant", "block", "trial"]
assert not main.duplicated(key).any(), "same participant/block/trial appears twice with different values - inspect manually"

per_p = main.groupby("participant").size()
per_pc = main.groupby(["participant", "condition"]).size()
print("Participants:", main["participant"].nunique(), "| trials per participant:", sorted(per_p.unique()))
print("Trials per participant x condition:", sorted(per_pc.unique()))

assert set(main["participant"]) == set(ROSTER), "participant list differs from the roster"
assert (per_p == N_BLOCKS * TRIALS_PER_BLOCK).all(), "some participant does not have exactly 24 trials"
assert (per_pc == TRIALS_PER_BLOCK).all()
assert set(main["condition"]) == set(CONDITIONS)
print("Structure OK: 16 x 24 =", len(main), "trials")

Participants: 16 | trials per participant: [np.int64(24)]
Trials per participant x condition: [np.int64(6)]
Structure OK: 16 x 24 = 384 trials


**Missing values** are only a problem when they are *unexpected*. By design, a trial that timed out or was interrupted has
no retrieval time - that blank means "not measured", not "data lost". So missingness is checked **per outcome status**.

In [6]:
perf_cols = ["retrieval_time_s", "total_tab_switches", "incorrect_selections", "first_selection_correct"]

print("Trials by outcome:")
print(main["controller_status"].value_counts().to_string(), "\n")

# Expected blanks: non-completed trials have no retrieval time.
completed = main["controller_status"] == "completed_auto"
unexpected = main.loc[completed, perf_cols].isna().sum()
print("Unexpected missing values among COMPLETED trials:")
print(unexpected.to_string())
assert unexpected.sum() == 0, "completed trials with missing performance values - inspect"

# Internal consistency of completed trials (the controller definitions in DATA_DICTIONARY.md)
c = main[completed]
assert (c["total_tab_switches"] == c["incorrect_selections"] + 1).all()
assert (c["first_selection_correct"] == (c["incorrect_selections"] == 0)).all()
assert c["retrieval_time_s"].between(0, 20).all()
print("Completed trials are internally consistent")

Trials by outcome:
controller_status
completed_auto    366
timeout            17
interrupted         1 

Unexpected missing values among COMPLETED trials:
retrieval_time_s           0
total_tab_switches         0
incorrect_selections       0
first_selection_correct    0
Completed trials are internally consistent


**Analysis flags.** Instead of deleting timeouts and interruptions, mark them. Retrieval time is only analysed for
`completed_auto` trials; timeouts are kept (they are a result: the participant failed within 20 s) and counted per condition later.

In [7]:
main["rt_valid"] = main["controller_status"].eq("completed_auto")
main["timed_out"] = main["controller_status"].eq("timeout")
main["interrupted"] = main["controller_status"].eq("interrupted")
main["accuracy_valid"] = main["first_selection_correct"].notna()   # has an observable first selection

note("main", "timeout trials (no retrieval time by design)", int(main["timed_out"].sum()), "kept, flagged timed_out")
note("main", "interrupted trials", int(main["interrupted"].sum()),
     f"kept, flagged interrupted ({int(main['technical_loss'].sum())} are technical loss)")

# Columns that are not needed downstream (still available in main_raw if ever required)
main_clean = main.drop(columns=["session_id", "mode", "measurement_source", "target_title", "source_file", "notes"])
print("main_clean:", main_clean.shape)
main_clean.head(3)

[main] timeout trials (no retrieval time by design): 17 -> kept, flagged timed_out
[main] interrupted trials: 1 -> kept, flagged interrupted (0 are technical loss)
main_clean: (384, 24)


,participant,group,block,condition,orientation,tab_load,content_set,trial,target_id,is_repeat,target_task_position,task_tab_count,visible_tab_count,controller_status,retrieval_time_s,total_tab_switches,incorrect_selections,first_selection_correct,no_selection,technical_loss,rt_valid,timed_out,interrupted,accuracy_valid
0,DUMMY001,2,1,H-H,H,high,C,T1,P3,0,7,15,16,completed_auto,10.304774,3.0,2.0,False,False,False,True,False,False,True
1,DUMMY001,2,1,H-H,H,high,C,T2,P4,0,10,15,16,completed_auto,10.523085,1.0,0.0,True,False,False,True,False,False,True
2,DUMMY001,2,1,H-H,H,high,C,T3,P5,0,13,15,16,completed_auto,15.109607,3.0,2.0,False,False,False,True,False,False,True


### 2.2 Block NASA-TLX (participant x condition level)
Fixes: long Google-Form headers -> short names, participant code standardised, condition label -> code (`"V-H: Vertical High"` -> `V-H`),
ratings **0-10 -> 0-100** (x10, as decided), duplicate submissions resolved, 4 blocks per participant verified.

In [8]:
blk = block_raw.copy()

# Long question text -> short names, matched by the dimension word the header starts with
def short_name(col):
    for dim, new in [("Mental", "mental_demand"), ("Physical", "physical_demand"), ("Temporal", "temporal_demand"),
                     ("Performance", "performance"), ("Effort", "effort"), ("Frustration", "frustration")]:
        if col.startswith(dim):
            return new
    return {"Timestamp": "timestamp",
            "What is your participant code? (e.g. P001)": "participant",
            "What condition block did you just finish?": "condition_label",
            "Optional: Any thoughts you would like to share in related to this condition?": "comment"}.get(col, col)

blk = blk.rename(columns=short_name).drop(columns=["Score"])
assert [c for c in TLX_DIMS if c not in blk.columns] == [], "a TLX dimension column was not recognised"

# Participant code: trim spaces, upper-case ('dummy007' -> 'DUMMY007')
fixed = blk["participant"].astype(str)
blk["participant"] = fixed.str.strip().str.upper()
note("block", "participant codes needing trim/upper-casing", int((fixed != blk["participant"]).sum()), "standardised")

# Condition code from the label
blk["condition"] = blk["condition_label"].str.split(":").str[0].str.strip()
assert set(blk["condition"]) <= set(CONDITIONS), set(blk["condition"])
blk = blk.drop(columns="condition_label")

# Codes that are not in the roster (typos) must be looked at by a human
unknown = sorted(set(blk["participant"]) - set(ROSTER))
note("block", "participant codes not in roster", len(unknown), (f"{unknown} -> fix by hand" if unknown else "none"))
assert not unknown

# Ratings are 0-10 on the form -> scale to 0-100
assert blk[TLX_DIMS].apply(lambda s: s.between(0, 10)).all().all(), "rating outside 0-10"
blk[TLX_DIMS] = blk[TLX_DIMS] * 10
blk["timestamp"] = pd.to_datetime(blk["timestamp"], format="%m/%d/%Y %H:%M:%S")

[block] participant codes needing trim/upper-casing: 1 -> standardised
[block] participant codes not in roster: 0 -> none


In [9]:
# Duplicate submissions for the same participant + condition: keep the latest, log it
dups = blk.duplicated(["participant", "condition"], keep=False)
if dups.any():
    print(blk[dups].sort_values(["participant", "condition", "timestamp"]).to_string())
blk = blk.sort_values("timestamp").drop_duplicates(["participant", "condition"], keep="last")
note("block", "duplicate participant+condition submissions", int(dups.sum()), "kept latest only" if dups.any() else "none")

# Completeness: expect 16 x 4 = 64
cnt = blk.groupby("participant")["condition"].nunique()
short = cnt[cnt < N_BLOCKS]
note("block", "participants with fewer than 4 conditions rated", len(short), "reported" if len(short) else "none")
assert len(blk) == N_PARTICIPANTS * N_BLOCKS, len(blk)

blk["raw_tlx"] = blk[TLX_DIMS].mean(axis=1)           # unweighted TLX (average of the 6 subscales)
block_clean = blk.drop(columns=["comment"]).sort_values(["participant", "condition"]).reset_index(drop=True)
print("block_clean:", block_clean.shape)
block_clean.head(3)

[block] duplicate participant+condition submissions: 0 -> none
[block] participants with fewer than 4 conditions rated: 0 -> none
block_clean: (64, 10)


,timestamp,participant,mental_demand,physical_demand,temporal_demand,performance,effort,frustration,condition,raw_tlx
0,2026-11-02 09:07:39,DUMMY001,60,50,40,40,70,70,H-H,55.000000
1,2026-11-02 09:17:43,DUMMY001,0,20,30,10,50,20,H-L,21.666667
2,2026-11-02 09:27:16,DUMMY001,20,40,50,30,80,60,V-H,46.666667


### 2.3 Weighted NASA-TLX (participant level)
Each participant answered 15 "which matters more" questions. A dimension's **weight = number of times it was chosen (0-5)**;
the 15 weights must add to 15. A participant who did not submit the form simply has no row - that is reported, never invented.

In [10]:
wt = weighted_raw.copy()
wt = wt.rename(columns={"What is your participant code?": "participant", "Timestamp": "timestamp"}).drop(columns=["Score"])
wt["participant"] = wt["participant"].astype(str).str.strip().str.upper()

def to_dim(text):
    """'Performance Demand' (typo in the form) and 'Frustration' -> clean dimension keys."""
    t = str(text).strip().strip(".").lower()
    for dim in TLX_DIMS:
        if t.startswith(dim.split("_")[0]):
            return dim
    raise ValueError(f"unrecognised dimension: {text!r}")

# One column per pairwise question: "Mental Demand vs. Physical Demand" -> (mental_demand, physical_demand)
pair_cols = {c: tuple(to_dim(p) for p in re.split(r"\s+vs\.?\s+", c)) for c in wt.columns if " vs" in c}
assert len(pair_cols) == 15, f"expected 15 pair columns, found {len(pair_cols)}"

# Every answer must be one of the two options in its own question
for col, (a, b) in pair_cols.items():
    chosen = wt[col].map(to_dim)
    assert chosen.isin([a, b]).all(), f"answer outside the pair in column {col!r}"
    wt[col] = chosen

weights = pd.DataFrame({d: sum((wt[c] == d).astype(int) for c in pair_cols) for d in TLX_DIMS})
weights.insert(0, "participant", wt["participant"])
assert (weights[TLX_DIMS].sum(axis=1) == 15).all(), "weights must sum to 15"

assert not weights["participant"].duplicated().any(), "participant submitted the weighting form twice"
assert set(weights["participant"]) <= set(ROSTER)
missing_w = sorted(set(ROSTER) - set(weights["participant"]))
note("weighted", "participants who did not submit the weighting form", len(missing_w), f"{missing_w} -> weighted TLX left blank (not imputed)")

weights_clean = weights.rename(columns={d: f"w_{d}" for d in TLX_DIMS}).sort_values("participant").reset_index(drop=True)
print("weights_clean:", weights_clean.shape)
weights_clean.head(3)

[weighted] participants who did not submit the weighting form: 1 -> ['DUMMY013'] -> weighted TLX left blank (not imputed)
weights_clean: (15, 7)


,participant,w_mental_demand,w_physical_demand,w_temporal_demand,w_performance,w_effort,w_frustration
0,DUMMY001,3,3,1,2,5,1
1,DUMMY002,2,3,4,1,4,1
2,DUMMY003,4,0,3,3,4,1


### 2.4 Screening / background (participant level)
Privacy: the form collects **names**. They are dropped here, immediately, so they never travel into the merged files.
Rows without a participant code are people who answered the screening form but were not enrolled.

In [11]:
scr = screen_raw.copy()
scr.columns = ["timestamp", "consent", "name", "age", "uses_browser_desktop", "desktop_browse_freq",
               "chrome_main", "multi_tab_freq", "window_habit", "usual_orientation", "participant"]
assert len(screen_raw.columns) == len(scr.columns)

scr = scr.drop(columns=["name"])                         # personal data - never carried forward
scr["participant"] = scr["participant"].astype("string").str.strip().str.upper()

no_code = scr["participant"].isna() | (scr["participant"] == "")
note("screening", "responses without a participant code (not enrolled)", int(no_code.sum()), "dropped")
scr = scr[~no_code]

assert not scr["participant"].duplicated().any(), "same participant screened twice"
assert set(scr["participant"]) == set(ROSTER)

# Consent should be 'I agree' for everyone who took part
assert scr["consent"].str.contains("agree", case=False).all()
screen_clean = scr.drop(columns=["timestamp", "consent"]).sort_values("participant").reset_index(drop=True)
print("screen_clean:", screen_clean.shape)
screen_clean.head(3)

[screening] responses without a participant code (not enrolled): 1 -> dropped
screen_clean: (16, 8)


,age,uses_browser_desktop,desktop_browse_freq,chrome_main,multi_tab_freq,window_habit,usual_orientation,participant
0,29,Yes,Daily,Yes,Sometimes,Usually in a single window,Horizontal,DUMMY001
1,24,Yes,Daily,No,Almost always,Usually in a single window,Horizontal,DUMMY002
2,25,Yes,Daily,Yes,Almost always,Usually in a single window,Horizontal,DUMMY003


## 3. Cross-source check: who is present where?
Before merging, confirm every participant appears in every source. Gaps are visible here instead of being hidden by a join.

In [12]:
coverage = pd.DataFrame({"participant": ROSTER}).set_index("participant")
coverage["main_trials"] = main_clean.groupby("participant").size()
coverage["block_ratings"] = block_clean.groupby("participant").size()
coverage["weighted_form"] = coverage.index.isin(weights_clean["participant"])
coverage["screening"] = coverage.index.isin(screen_clean["participant"])
print(coverage.to_string())

incomplete = coverage[(coverage["main_trials"] != 24) | (coverage["block_ratings"] != 4) | ~coverage["weighted_form"] | ~coverage["screening"]]
print("\nParticipants with an incomplete record:", incomplete.index.tolist())

             main_trials  block_ratings  weighted_form  screening
participant                                                      
DUMMY001              24              4           True       True
DUMMY002              24              4           True       True
DUMMY003              24              4           True       True
DUMMY004              24              4           True       True
DUMMY005              24              4           True       True
DUMMY006              24              4           True       True
DUMMY007              24              4           True       True
DUMMY008              24              4           True       True
DUMMY009              24              4           True       True
DUMMY010              24              4           True       True
DUMMY011              24              4           True       True
DUMMY012              24              4           True       True
DUMMY013              24              4          False       True
DUMMY014  

## 4. Combine at the right grain
Two analysis tables, matching the two analysis levels:

- **`trial_level`** (384 rows) - for models of single trials (e.g. retrieval time with participant as a random effect). Participant background is attached.
- **`condition_level`** (64 rows) - one row per participant x condition: trial performance summarised + block TLX + weighted TLX. This is the table for the 2x2 repeated-measures tests.

`validate="many_to_one"` / `"one_to_one"` on each merge makes pandas raise an error if a key is duplicated, and the row-count asserts catch any join that adds or loses rows.

In [13]:
# ---- trial_level: trials + participant background -----------------------------------------
trial_level = main_clean.merge(screen_clean, on="participant", how="left", validate="many_to_one")
assert len(trial_level) == len(main_clean) == 384
trial_level = trial_level.sort_values(["participant", "block", "trial"]).reset_index(drop=True)
print("trial_level:", trial_level.shape)

trial_level: (384, 31)


In [14]:
# ---- condition_level: summarise trials first, then join the one-row-per-condition survey data -------------
# Rules (match paper 4.7):
#   - RT, incorrect selections: COMPLETED trials only.
#   - First-selection accuracy: every trial with an observable first selection (completed trials AND timeouts
#     where the participant did click something). Trials with no observable first selection are excluded.
#   - Timeouts / no-selection trials are counted separately so they can be reported next to the averages.
g = main_clean.groupby(["participant", "condition", "orientation", "tab_load"])
perf = g.apply(lambda d: pd.Series({
    "n_trials": len(d),
    "n_completed": int(d["rt_valid"].sum()),
    "n_timeouts": int(d["timed_out"].sum()),
    "n_interrupted": int(d["interrupted"].sum()),
    "n_no_selection": int(d["no_selection"].fillna(False).sum()),
    "n_accuracy_valid": int(d["first_selection_correct"].notna().sum()),
    "mean_rt_s": d.loc[d["rt_valid"], "retrieval_time_s"].mean(),
    "mean_incorrect_selections": d.loc[d["rt_valid"], "incorrect_selections"].mean(),
    "first_try_accuracy": d["first_selection_correct"].dropna().astype(float).mean(),
}), include_groups=False).reset_index()
assert len(perf) == 64
int_cols = ["n_trials", "n_completed", "n_timeouts", "n_interrupted", "n_no_selection", "n_accuracy_valid"]
perf[int_cols] = perf[int_cols].astype(int)

cond = perf.merge(block_clean.drop(columns="timestamp"), on=["participant", "condition"], how="left", validate="one_to_one")
cond = cond.merge(weights_clean, on="participant", how="left", validate="many_to_one")
cond = cond.merge(screen_clean, on="participant", how="left", validate="many_to_one")

# Weighted TLX = sum(weight x rating) / 15, on the 0-100 scale. Blank if the participant has no weights.
wcols = [f"w_{d}" for d in TLX_DIMS]
cond["weighted_tlx"] = (cond[wcols].to_numpy() * cond[TLX_DIMS].to_numpy()).sum(axis=1) / 15
cond.loc[cond[wcols].isna().any(axis=1), "weighted_tlx"] = np.nan

condition_level = cond.sort_values(["participant", "condition"]).reset_index(drop=True)
assert len(condition_level) == 64
print("condition_level:", condition_level.shape)
condition_level.head(4)

condition_level: (64, 34)


,participant,condition,orientation,tab_load,n_trials,n_completed,n_timeouts,n_interrupted,n_no_selection,n_accuracy_valid,mean_rt_s,mean_incorrect_selections,first_try_accuracy,mental_demand,physical_demand,temporal_demand,performance,effort,frustration,raw_tlx,w_mental_demand,w_physical_demand,w_temporal_demand,w_performance,w_effort,w_frustration,age,uses_browser_desktop,desktop_browse_freq,chrome_main,multi_tab_freq,window_habit,usual_orientation,weighted_tlx
0,DUMMY001,H-H,H,high,6,6,0,0,0,6,9.826922,0.833333,0.500000,60,50,40,40,70,70,55.000000,3.0,3.0,1.0,2.0,5.0,1.0,29,Yes,Daily,Yes,Sometimes,Usually in a single window,Horizontal,58.000000
1,DUMMY001,H-L,H,low,6,6,0,0,0,6,3.478273,0.666667,0.666667,0,20,30,10,50,20,21.666667,3.0,3.0,1.0,2.0,5.0,1.0,29,Yes,Daily,Yes,Sometimes,Usually in a single window,Horizontal,25.333333
2,DUMMY001,V-H,V,high,6,6,0,0,0,6,10.144269,0.833333,0.333333,20,40,50,30,80,60,46.666667,3.0,3.0,1.0,2.0,5.0,1.0,29,Yes,Daily,Yes,Sometimes,Usually in a single window,Horizontal,50.000000
3,DUMMY001,V-L,V,low,6,6,0,0,0,6,6.076741,0.000000,1.000000,0,30,30,10,20,40,21.666667,3.0,3.0,1.0,2.0,5.0,1.0,29,Yes,Daily,Yes,Sometimes,Usually in a single window,Horizontal,18.666667


## 5. Final validation and export

In [15]:
assert trial_level.groupby("participant").size().eq(24).all()
assert condition_level.groupby("participant").size().eq(4).all()
assert condition_level["n_trials"].eq(TRIALS_PER_BLOCK).all()
assert not trial_level.duplicated(["participant", "block", "trial"]).any()
assert not condition_level.duplicated(["participant", "condition"]).any()
assert "name" not in trial_level.columns and "name" not in condition_level.columns   # no personal data in outputs

print("Remaining missing values in condition_level (expected: only weighted TLX for participants without the form,")
print("and RT/accuracy if a condition had no completed trial):")
m = condition_level.isna().sum(); print(m[m > 0].to_string())

OUT_DIR.mkdir(exist_ok=True)
trial_level.to_csv(OUT_DIR / "trial_level.csv", index=False)
condition_level.to_csv(OUT_DIR / "condition_level.csv", index=False)
cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(OUT_DIR / "cleaning_log.csv", index=False)
print("\nSaved trial_level.csv, condition_level.csv, cleaning_log.csv to", OUT_DIR.resolve())
cleaning_log

Remaining missing values in condition_level (expected: only weighted TLX for participants without the form,
and RT/accuracy if a condition had no completed trial):
w_mental_demand      4
w_physical_demand    4
w_temporal_demand    4
w_performance        4
w_effort             4
w_frustration        4
weighted_tlx         4



Saved trial_level.csv, condition_level.csv, cleaning_log.csv to /tmp/claude-0/-home-claude/714ea817-ab60-5452-9e7e-d7ecdff4929d/scratchpad/hs/out/v3/cleaned


,stage,issue,n_affected,action
0,main,rows with mode != 'test',0,dropped
1,main,exact duplicate rows,1,dropped (kept first)
2,main,timeout trials (no retrieval time by design),17,"kept, flagged timed_out"
3,main,interrupted trials,1,"kept, flagged interrupted (0 are technical loss)"
4,block,participant codes needing trim/upper-casing,1,standardised
5,block,participant codes not in roster,0,none
6,block,duplicate participant+condition submissions,0,none
7,block,participants with fewer than 4 conditions rated,0,none
8,weighted,participants who did not submit the weighting ...,1,['DUMMY013'] -> weighted TLX left blank (not i...
9,screening,responses without a participant code (not enro...,1,dropped


In [16]:
# Quick sanity look at the design cells (descriptives only - no inference yet)
condition_level.groupby(["orientation", "tab_load"])[["mean_rt_s", "first_try_accuracy", "n_timeouts", "n_no_selection", "raw_tlx", "weighted_tlx"]].mean().round(2)

mean_rt_s  first_try_accuracy  n_timeouts  n_no_selection  raw_tlx  weighted_tlx
orientation tab_load                                                                                  
H           high           7.57                0.31        0.75            0.25    50.94         53.42
            low            3.69                0.72        0.06            0.00    21.67         22.22
V           high           6.42                0.49        0.19            0.06    46.15         47.11
            low            3.65                0.72        0.06            0.00    22.81         23.56